## Homework 4

In [2]:
import pandas as pd
import numpy as np

In [3]:

train = pd.read_csv("playground-series-s6e4/train.csv")
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [8]:

from pycaret.classification import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# train/test split from original dataset
train_pc, test_pc = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["Irrigation_Need"]
)

# setup experiment
clf_setup = setup(
    data=train_pc,
    target="Irrigation_Need",
    session_id=42,
    fold_strategy="stratifiedkfold",
    fold=5,
    numeric_imputation="median",
    categorical_imputation="mode",
    normalize=False,
    verbose=False
)

# register balanced accuracy metric for versions where it is not built in
add_metric(
    id="balanced_acc",
    name="Balanced Accuracy",
    score_func=balanced_accuracy_score,
    greater_is_better=True
)

# compare top models, excluding CatBoost, ranked by balanced accuracy
top_models = compare_models(
    n_select=3,
    exclude=["catboost"],
    sort="balanced_acc"
)

# tune the best model by balanced accuracy
tuned_model = tune_model(
    top_models[0],
    optimize="balanced_acc"
)

# blend top models
blended_model = blend_models(estimator_list=top_models)

# stack top models
stacked_model = stack_models(estimator_list=top_models)

# finalize chosen model
final_model = finalize_model(stacked_model)

# predict on held-out test set
preds = predict_model(final_model, data=test_pc)

# show results
preds.head()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9835,0.9965,0.9835,0.9834,0.9834,0.9675,0.9676,0.9597,3.6260
gbc,Gradient Boosting Classifier,0.9843,0.0000,0.9843,0.9843,0.9842,0.9691,0.9692,0.9565,59.4280
rf,Random Forest Classifier,0.9843,0.9959,0.9843,0.9842,0.9841,0.9690,0.9691,0.9515,11.8800
dt,Decision Tree Classifier,0.9691,0.9718,0.9691,0.9691,0.9691,0.9394,0.9394,0.9439,2.4360
ada,Ada Boost Classifier,0.9557,0.0000,0.9557,0.9569,0.9551,0.9132,0.9141,0.8671,5.4060
et,Extra Trees Classifier,0.9364,0.9880,0.9364,0.9376,0.9338,0.8736,0.8741,0.7879,11.0960
lda,Linear Discriminant Analysis,0.8719,0.0000,0.8719,0.8759,0.8697,0.7476,0.7493,0.7196,2.2240
ridge,Ridge Classifier,0.8559,0.0000,0.8559,0.8285,0.8418,0.7098,0.7114,0.5852,1.6740
nb,Naive Bayes,0.7450,0.8135,0.7450,0.7421,0.7327,0.4713,0.4794,0.5590,1.7380
lr,Logistic Regression,0.7218,0.0000,0.7218,0.7117,0.7084,0.4307,0.4331,0.4920,11.2140


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy
Fold,,,,,,,,
0,0.9844,0.9972,0.9844,0.9844,0.9843,0.9693,0.9694,0.9594
1,0.9843,0.9972,0.9843,0.9842,0.9842,0.9690,0.9691,0.9603
2,0.9843,0.9973,0.9843,0.9843,0.9843,0.9692,0.9692,0.9616
3,0.9845,0.9972,0.9845,0.9844,0.9844,0.9694,0.9695,0.9602
4,0.9840,0.9969,0.9840,0.9839,0.9839,0.9685,0.9685,0.9604
Mean,0.9843,0.9971,0.9843,0.9843,0.9842,0.9691,0.9691,0.9604
Std,0.0002,0.0001,0.0002,0.0002,0.0002,0.0003,0.0003,0.0007


Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy
Fold,,,,,,,,
0,0.9847,0.9965,0.9847,0.9846,0.9846,0.9698,0.9699,0.9575
1,0.9845,0.9965,0.9845,0.9844,0.9844,0.9694,0.9695,0.9572
2,0.9854,0.9967,0.9854,0.9854,0.9854,0.9714,0.9714,0.9601
3,0.9856,0.9966,0.9856,0.9856,0.9855,0.9717,0.9717,0.9595
4,0.9850,0.9962,0.9850,0.9850,0.9849,0.9705,0.9706,0.9609
Mean,0.9850,0.9965,0.9850,0.9850,0.9850,0.9706,0.9706,0.9591
Std,0.0004,0.0002,0.0004,0.0004,0.0004,0.0009,0.0009,0.0015


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018001 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2980
[LightGBM] [Info] Number of data points in the train set: 282240, number of used features: 43
[LightGBM] [Info] Start training from score -3.400772
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038691 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2981
[LightGBM] [Info] Number of data points in the train set: 282240, number of used features: 43
[LightGBM] [Info] Start training from score -3.400772
[LightGBM] [Info] Start training from score -0.532434
[LightGBM] [Info] Start training from score -0.968957
[LightGBM] [Info] Auto-choosing 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy
Fold,,,,,,,,
0,0.9847,0.0000,0.9847,0.9847,0.9846,0.9699,0.9699,0.9589
1,0.9845,0.0000,0.9845,0.9844,0.9844,0.9694,0.9695,0.9583
2,0.9854,0.0000,0.9854,0.9854,0.9854,0.9713,0.9714,0.9614
3,0.9857,0.0000,0.9857,0.9857,0.9856,0.9719,0.9719,0.9610
4,0.9850,0.0000,0.9850,0.9849,0.9849,0.9704,0.9705,0.9623
Mean,0.9851,0.0000,0.9851,0.9850,0.9850,0.9706,0.9707,0.9604
Std,0.0005,0.0000,0.0005,0.0005,0.0005,0.0009,0.0009,0.0015


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008263 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2981
[LightGBM] [Info] Number of data points in the train set: 282240, number of used features: 43
[LightGBM] [Info] Start training from score -3.400772
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028087 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2981
[LightGBM] [Info] Number of data points in the train set: 282240, number of used features: 43
[LightGBM] [Info] Start training from score -3.400772
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing 

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy
0,Stacking Classifier,0.9861,0.9962,0.9861,0.9860,0.9860,0.9726,0.9727,0.9646


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,prediction_label,prediction_score
511403,511403,Clay,5.83,64.410004,0.39,1.78,13.220000,38.480000,1238.910034,6.00,...,Rabi,Drip,River,14.46,Yes,33.130001,East,Low,Low,0.9939
498032,498032,Loamy,6.25,24.480000,0.34,2.35,13.960000,56.900002,1691.410034,8.71,...,Rabi,Sprinkler,River,7.08,No,75.730003,Central,Medium,Medium,0.9818
200004,200004,Silt,6.63,58.400002,1.18,0.60,27.320000,53.930000,2058.209961,8.53,...,Zaid,Rainfed,Reservoir,8.66,No,19.540001,West,Medium,Medium,0.9900
383295,383295,Sandy,6.54,46.110001,0.76,3.12,39.369999,57.439999,1546.989990,10.64,...,Kharif,Sprinkler,River,13.85,Yes,108.320000,Central,Low,Low,0.9946
303295,303295,Clay,7.09,63.099998,1.29,0.57,33.770000,44.689999,1887.800049,5.07,...,Kharif,Drip,Reservoir,7.45,No,15.640000,North,Low,Low,0.9863
